<a href="https://colab.research.google.com/github/yijing461/4002-Project/blob/main/task1_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load and verify the dataset

In [ ]:
import os
import re
import json
import pickle
import tarfile
import zipfile
import urllib.request
from collections import Counter
import numpy as np

os.makedirs("data", exist_ok=True)

if not os.path.exists("data/en-US.jsonl"):
    urllib.request.urlretrieve("https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz", "data/massive.tar.gz")
    tar = tarfile.open("data/massive.tar.gz")
    content = tar.extractfile("1.0/data/en-US.jsonl").read()
    tar.close()
    with open("data/en-US.jsonl", "wb") as f:
        f.write(content)

In [ ]:
records = []
with open("data/en-US.jsonl", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

train_raw = [r for r in records if r["partition"] == "train"]
dev_raw = [r for r in records if r["partition"] == "dev"]
test_raw = [r for r in records if r["partition"] == "test"]

print("Total:", len(records))
print("Train:", len(train_raw))
print("Dev:", len(dev_raw))
print("Test:", len(test_raw))
assert len(train_raw) == 11514
assert len(dev_raw) == 2033
assert len(test_raw) == 2974

Total: 16521
Train: 11514
Dev: 2033
Test: 2974


## Implement preprocessing

In [ ]:
def parse_annotation(annot):
    tokens = []
    tags = []
    for part in re.split(r"(\[[^\]]+\])", annot):
        if part.startswith("["):
            slot_type, span = part[1:-1].split(" : ", 1)
            for i, word in enumerate(span.split()):
                tokens.append(word)
                if i == 0:
                    tags.append("B-" + slot_type)
                else:
                    tags.append("I-" + slot_type)
        else:
            for word in part.split():
                tokens.append(word)
                tags.append("O")
    return tokens, tags


def get_spans(tokens, tags):
    spans = []
    slot_type = None
    words = []
    for word, tag in zip(tokens, tags):
        if tag.startswith("I-") and tag[2:] == slot_type:
            words.append(word)
            continue
        if slot_type is not None:
            spans.append((slot_type, " ".join(words)))
        slot_type = None
        words = []
        if tag != "O":
            slot_type = tag[2:]
            words = [word]
    if slot_type is not None:
        spans.append((slot_type, " ".join(words)))
    return spans


def process(raw):
    output = []
    for r in raw:
        tokens, tags = parse_annotation(r["annot_utt"])
        slots = re.findall(r"\[(\S+) : ([^\]]+)\]", r["annot_utt"])
        assert tokens == r["utt"].split()
        assert get_spans(tokens, tags) == slots
        output.append({"id": r["id"], "utt": r["utt"], "tokens": tokens, "tags": tags, "intent": r["intent"], "slots": slots})
    return output


train_data = process(train_raw)
dev_data = process(dev_raw)
test_data = process(test_raw)

print("Checked", len(train_data) + len(dev_data) + len(test_data), "sentences")
print(train_data[0]["tokens"])
print(train_data[0]["tags"])
print(train_data[0]["slots"])

Checked 16521 sentences
['wake', 'me', 'up', 'at', 'nine', 'am', 'on', 'friday']
['O', 'O', 'O', 'O', 'B-time', 'I-time', 'O', 'B-date']
[('time', 'nine am'), ('date', 'friday')]


In [ ]:
MIN_FREQ = 1

word_counts = Counter()
for d in train_data:
    word_counts.update(d["tokens"])

word2id = {"<PAD>": 0, "<UNK>": 1}
for word in sorted(word_counts):
    if word_counts[word] >= MIN_FREQ:
        word2id[word] = len(word2id)

slot_types = set()
intents = set()
for d in train_data:
    intents.add(d["intent"])
    for tag in d["tags"]:
        if tag != "O":
            slot_types.add(tag[2:])

tag2id = {"O": 0}
for s in sorted(slot_types):
    tag2id["B-" + s] = len(tag2id)
    tag2id["I-" + s] = len(tag2id)

intent2id = {}
for intent in sorted(intents):
    intent2id[intent] = len(intent2id)

id2tag = {i: t for t, i in tag2id.items()}
id2intent = {i: t for t, i in intent2id.items()}

print("Vocabulary size:", len(word2id))
print("Slot types:", len(slot_types), "BIO tags:", len(tag2id))
print("Intents:", len(intent2id))
assert len(tag2id) == 111
assert len(intent2id) == 60

Vocabulary size: 5304
Slot types: 55 BIO tags: 111
Intents: 60


In [ ]:
def encode(data):
    max_len = max(len(d["tokens"]) for d in data)
    X = np.zeros((len(data), max_len), dtype=np.int64)
    Y_tags = np.full((len(data), max_len), -100, dtype=np.int64)
    Y_intent = np.zeros(len(data), dtype=np.int64)
    lengths = np.zeros(len(data), dtype=np.int64)
    for i, d in enumerate(data):
        for j in range(len(d["tokens"])):
            X[i, j] = word2id.get(d["tokens"][j], 1)
            Y_tags[i, j] = tag2id[d["tags"][j]]
        Y_intent[i] = intent2id[d["intent"]]
        lengths[i] = len(d["tokens"])
    return X, Y_tags, Y_intent, lengths


train_arrays = encode(train_data)
dev_arrays = encode(dev_data)
test_arrays = encode(test_data)

for name, arrays in [("Train", train_arrays), ("Dev", dev_arrays), ("Test", test_arrays)]:
    X, Y_tags, Y_intent, lengths = arrays
    unk = 100 * (X == 1).sum() / lengths.sum()
    print(name, "X:", X.shape, "Y_tags:", Y_tags.shape, "Y_intent:", Y_intent.shape, "unknown words:", round(unk, 2), "%")

Train X: (11514, 35) Y_tags: (11514, 35) Y_intent: (11514,) unknown words: 0.0 %
Dev X: (2033, 27) Y_tags: (2033, 27) Y_intent: (2033,) unknown words: 3.32 %
Test X: (2974, 61) Y_tags: (2974, 61) Y_intent: (2974,) unknown words: 3.51 %


In [ ]:
GLOVE_DIM = 300
glove_file = "data/glove.6B." + str(GLOVE_DIM) + "d.txt"

if not os.path.exists(glove_file):
    urllib.request.urlretrieve("https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip", "data/glove.6B.zip")
    with zipfile.ZipFile("data/glove.6B.zip") as z:
        z.extract("glove.6B." + str(GLOVE_DIM) + "d.txt", "data")

np.random.seed(42)
embedding_matrix = np.random.normal(0, 0.1, (len(word2id), GLOVE_DIM)).astype(np.float32)
embedding_matrix[0] = 0
found = 0
with open(glove_file, encoding="utf-8") as f:
    for line in f:
        values = line.rstrip().split(" ")
        if values[0] in word2id:
            embedding_matrix[word2id[values[0]]] = np.array(values[1:], dtype=np.float32)
            found += 1

print("Embedding matrix:", embedding_matrix.shape)
print("Words found in GloVe:", found, "of", len(word2id) - 2)
print("Embedding parameters:", embedding_matrix.size)

Embedding matrix: (5304, 300)
Words found in GloVe: 4918 of 5302
Embedding parameters: 1591200


In [ ]:
processed = {
    "train_data": train_data, "dev_data": dev_data, "test_data": test_data,
    "train_arrays": train_arrays, "dev_arrays": dev_arrays, "test_arrays": test_arrays,
    "word2id": word2id, "tag2id": tag2id, "id2tag": id2tag,
    "intent2id": intent2id, "id2intent": id2intent,
    "embedding_matrix": embedding_matrix,
}
with open("processed.pkl", "wb") as f:
    pickle.dump(processed, f)
print("Saved processed.pkl:", round(os.path.getsize("processed.pkl") / 1e6, 1), "MB")

Saved processed.pkl: 20.1 MB
